In [0]:
%run ./config

In [0]:
spark.sql(f"""
CREATE SCHEMA IF NOT EXISTS {catalog_name}.f_control
COMMENT 'Control schema for metadata-driven batch and streaming ingestion frameworks.'
""")
print(f"  Schema {catalog_name}.f_control ensured")


In [0]:

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog_name}.f_control.streaming_ingestion_metadata (

  -- Pipeline Identification
  batch_control_key        STRING    NOT NULL COMMENT 'Unique key for each streaming pipeline',
  source_system            STRING    NOT NULL COMMENT 'Source system publishing events (e.g. CSnx)',

  -- Event Hub Connection
  event_hub_namespace     STRING    NOT NULL COMMENT 'Azure Event Hub namespace',
  event_hub_name          STRING    NOT NULL COMMENT 'Event Hub name / Kafka topic',
  consumer_group           STRING    DEFAULT '$Default' COMMENT 'Kafka consumer group',
  event_hub_secret_key    STRING    COMMENT 'Key Vault secret name (fallback when MSI unavailable)',

  -- Starting Position
  starting_offset         STRING    DEFAULT '-1' COMMENT '-1 = earliest, @latest = end of stream',
  starting_seq_no         BIGINT    DEFAULT -1 COMMENT 'Starting sequence number (-1 = beginning)',
  starting_enqueued_time  STRING    COMMENT 'ISO 8601 starting enqueued time (NULL = not set)',
  starting_is_inclusive   BOOLEAN   DEFAULT true COMMENT 'Starting position inclusive (true) or exclusive',

  -- Message Format
  data_format             STRING    DEFAULT 'JSON' COMMENT 'Payload format: JSON, XML, CSV, AVRO',
  xml_row_tag             STRING    COMMENT 'XML row delimiter tag (required when data_format)',

  -- Streaming Settings
  max_events_per_trigger  INT       DEFAULT 10000 COMMENT 'Max events per micro-batch trigger',
  trigger_interval        STRING    DEFAULT '30 seconds' COMMENT 'Structured Streaming processingTime interval',

  -- Target Tables
  checkpoint_location     STRING    NOT NULL COMMENT 'ABFSS/Volume path for streaming checkpoint (unique)',
  bronze_table_name       STRING    NOT NULL COMMENT 'Fully qualified bronze Delta table',
  silver_table_name       STRING    COMMENT 'Fully qualified silver Delta table',
  silver_custom_notebook_url STRING COMMENT 'Workspace path to custom bronze-to-silver notebook',
  partition_column        STRING    COMMENT 'Column to partition by (e.g. ingestion_date)',
  gold_table_name         STRING    COMMENT 'Fully qualified gold Delta table',
  gold_custom_notebook_url   STRING COMMENT 'Workspace path to custom silver-to-gold notebook',

  -- Data Quality
  dq_bronze_rules         STRING    COMMENT 'JSON array of bronze DQ rules',
  dq_silver_rules         STRING    COMMENT 'JSON array of silver DQ rules',
  gold_key_column         STRING    COMMENT 'Key column for gold layer dedup/SCD',
  enable_exactly_once     BOOLEAN   DEFAULT true COMMENT 'Use foreachBatch + MERGE for exactly-once delivery',

  -- Status
  is_active               BOOLEAN   DEFAULT true COMMENT 'Enable/disable this streaming pipeline',

  -- Audit
  created_by              STRING    DEFAULT current_user() COMMENT 'User who created this record',
  created_timestamp       TIMESTAMP DEFAULT current_timestamp() COMMENT 'Record creation timestamp',
  updated_by              STRING    COMMENT 'User who last updated this record',
  updated_timestamp       TIMESTAMP COMMENT 'Timestamp of last update'
)
USING DELTA
COMMENT 'Control table for the streaming ingestion framework. Each row defines a complete streaming pipeline'
TBLPROPERTIES (
  'delta.autoOptimize.optimizeWrite' = 'true',
  'delta.autoOptimize.autoCompact'   = 'true',
  'delta.feature.allowColumnDefaults' = 'supported',
  'delta.columnMapping.mode'          = 'name'
)
""")
print(f"  Table {catalog_name}.f_control.streaming_ingestion_metadata ensured")


In [0]:

try:
    spark.sql(f"""
    ALTER TABLE {catalog_name}.f_control.streaming_ingestion_metadata
    ADD CONSTRAINT pk_streaming_ingestion_metadata PRIMARY KEY (batch_control_key)
    """)
    print("  Primary key constraint added")
except Exception as e:
    if "already exists" in str(e).lower() or "CONSTRAINT_ALREADY_EXISTS" in str(e):
        print("  Primary key constraint already exists \u2014 skipping")
    else:
        raise

display(
    spark.read.table(f"{catalog_name}.f_control.streaming_ingestion_metadata")
    .limit(20)
)

In [0]:
spark.sql(f"""
INSERT INTO {catalog_name}.f_control.streaming_ingestion_metadata
(
  batch_control_key,
  source_system,
  event_hub_namespace,
  event_hub_name,
  consumer_group,
  event_hub_secret_key,
  starting_offset,
  starting_seq_no,
  starting_enqueued_time,
  starting_is_inclusive,
  data_format,
  xml_row_tag,
  max_events_per_trigger,
  trigger_interval,
  checkpoint_location,
  bronze_table_name,
  silver_table_name,
  partition_column,
  gold_table_name,
  enable_exactly_once,
  is_active
)
VALUES
(
  'streaming_sales',
  'CSnx',
  'pkc-619z3.us-east1.gcp.confluent.cloud',
  'xml_topic',
  '$Default',
  'kafka-api-key',
  '-1',
  -1,
  NULL,
  true,
  'XML',
  'order',
  10000,
  '30 seconds',
  '/Volumes/streaming_dev/landing_vol/kafka/checkpoint/sales',
  'streaming_dev.bronze.b_sales',
  'streaming_dev.silver.s_sales',
  'ingestion_date',
  'streaming_dev.gold.g_sales',
  true,
  true
)
""")
print(f"  Dummy row inserted into {catalog_name}.f_control.streaming_ingestion_metadata")

display(
    spark.read.table(f"{catalog_name}.f_control.streaming_ingestion_metadata")
    .limit(20)
)